# Map integration

In [25]:
import folium
from folium.plugins import MarkerCluster
import pandas as pd
import numpy as np
import webbrowser
from scipy.sparse import csr_matrix
import networkx as nx

In [2]:
edges = pd.read_csv("../data/edges.csv").rename(columns=lambda x: x.strip())[["# source", "target", "distance","airline"]].drop_duplicates()
nodes = pd.read_csv("../data/nodes.csv").rename(columns=lambda x: x.strip())
nodes = nodes[["id", "name","city","country","latitude","longitude","altitude"]]

In [3]:
nodes.head()

,id,name,city,country,latitude,longitude,altitude
0,1,Goroka Airport,Goroka,Papua New Guinea,-6.081690,145.391998,5282
1,2,Madang Airport,Madang,Papua New Guinea,-5.207080,145.789001,20
2,3,Mount Hagen Kagamuga Airport,Mount Hagen,Papua New Guinea,-5.826790,144.296005,5388
3,4,Nadzab Airport,Nadzab,Papua New Guinea,-6.569803,146.725977,239
4,5,Port Moresby Jacksons International Airport,Port Moresby,Papua New Guinea,-9.443380,147.220001,146


In [15]:
class Map:
    def __init__(self, center, zoom_start):
        self.center = center
        self.zoom_start = zoom_start
        # Mapu inicializujeme hneď, aby sme do nej mohli pridávať uzly
        self.m = folium.Map(location=self.center, zoom_start=self.zoom_start, tiles="cartodb positron",prefer_canvas=True)
        self.nodes = None
    
    def add_nodes(self, nodes):
        marker_cluster = MarkerCluster().add_to(self.m)
        for i, row in nodes.iterrows():
            # Pridanie popupu s názvom (ak ho máte v stĺpci "nazov")
            folium.Marker(
                location=[row["latitude"], row["longitude"]],
                popup=row["name"] 
            ).add_to(marker_cluster)
        self.nodes = nodes
    
    def add_flights(self, flights_df, color="blue", weight=2):
        for i, row in flights_df.iterrows():
            # Definícia bodov letu
            start = [float(x) for x in row["start"].split(",")]
            end = [float(x) for x in row["end"].split(",")]
            # Vykreslenie čiary medzi bodmi
            folium.PolyLine(
                locations=[start, end],
                color=color,
                weight=weight,
                opacity=0.6,
                tooltip=row.get("flight_no", "Let")
            ).add_to(self.m)
    
    def showMap(self, open):
        # Uloženie a otvorenie robíme až na záver
        self.m.save("map.html")
        if open:
            webbrowser.open("map.html")

# --- Použitie ---
# map_obj = Map([48.14, 17.10], 7)
# map_obj.add_nodes(moje_df)
# map_obj.showMap()


In [202]:
attractive = edges.groupby(["# source","target"])["airline"].count().sort_values()
attractive = attractive[attractive >7].index.to_list()

In [19]:
slovakia_airports = nodes[nodes.country == "Germany"].index.to_list()
slovakia_edges = edges.query("`# source` in @slovakia_airports or target in @slovakia_airports")

In [20]:
slovakia_edges.count()

# source    4476
target      4476
distance    4476
airline     4476
dtype: int64

In [212]:
edges_attractive = edges[edges[["# source", "target"]].apply(lambda row: (row["# source"], row["target"]) in attractive, axis=1)]

In [21]:
slovakia_edges[["start","end"]] = slovakia_edges[["# source", "target"]].apply(lambda row: 
                                           f"{nodes.loc[row['# source'],'latitude']}," +
                                           f"{nodes.loc[row['# source'],'longitude']};"+
                                           f"{nodes.loc[row['target'],'latitude']},"+
                                           f"{nodes.loc[row['target'],'longitude']}", axis=1).str.split(";",expand=True)

In [22]:
mapa = Map((48.14, 17.10), zoom_start=5)
mapa.add_nodes(nodes)
mapa.add_flights(slovakia_edges)
mapa.showMap(True)

In [57]:
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    R = 6371.0  # Radius of the Earth in km
    
    # Convert degrees to radians
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    
    # Haversine formula
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1-a))
    
    return R * c

In [ ]:
calculate_distance(50.1008,14.26,48.17020034790039,17.21269989013672)

303.64758166571426

In [26]:
A = np.array([[0 for i in range(3214)] for i in range(3214)])


for source, target, distance in edges.values:
    A[np.int16(source)][np.int16(target)] = distance

G = nx.from_numpy_array(A, create_using=nx.DiGraph)

ValueError: too many values to unpack (expected 3)